# Policy composition, activation weight, and normalized payoff

Rows show population sizes and columns show agent groups. The horizontal axis is each random agent's activation weight relative to a unit-weight payoff-guided agent, the vertical axis is the realized payoff-guided agent count, and color is expected payoff at the common normalized-time endpoint. Each count row is smoothed independently over activation weight with the fixed Gaussian kernel shown in the bottom-right glyph, drawn at the same horizontal scale as a heatmap panel. The color scale runs from the empty-network reference at zero to the star reference at one, with negative values shown in the dark-blue under-range region. Composition is sampled independently and evenly over all realizable agent counts; the remaining parameters use the configured space-filling sample.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib_inline.backend_inline import set_matplotlib_formats
import numpy as np
import scienceplots  # noqa: F401
import xarray as xr

plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "figure.titlesize": 20,
})

project_root = Path.cwd().resolve()
if not (project_root / "tasks").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tasks.policies import load_policy_results

In [ ]:
result_store_path = project_root / "artifacts/simulation_runs/run-20260728T232448.zarr"
ds = load_policy_results(
    result_store_path,
    stats=["policy_welfare.mean"],
    normalize_time=True,
)
if ds.attrs["has_failures"]:
    raise RuntimeError(f"Run contains failed rows: {result_store_path}")
print(
    f"Loaded {result_store_path} at "
    f"{float(ds.step[-1]):g} activation opportunities per agent"
)

In [ ]:
policy_payoff = ds["policy_welfare.mean"].isel(step=-1, drop=True)
best_response_payoff = policy_payoff.sel(policy_class="best_response", drop=True)
random_payoff = policy_payoff.sel(policy_class="random", drop=True)
best_response_count = ds.coords["best_response_count"]
random_count = ds.coords["random_count"]
n_agents = ds.coords["n_agents"]
everyone_payoff = (
    xr.where(best_response_count > 0, best_response_count * best_response_payoff, 0.0)
    + xr.where(random_count > 0, random_count * random_payoff, 0.0)
) / n_agents

population_groups = ["All agents", "Payoff-guided agents", "Random-action agents"]
group_payoff = xr.concat(
    [everyone_payoff, best_response_payoff, random_payoff],
    dim=xr.IndexVariable("population_group", population_groups),
).rename("mean_normalized_payoff")
random_activation_weight = ds["theta"].sel(
    theta_parameter="random.agent.activation_weight",
    drop=True,
).rename("random_activation_weight")
plot_frame = xr.Dataset({
    "mean_normalized_payoff": group_payoff,
    "random_activation_weight": random_activation_weight,
}).to_dataframe().reset_index()
plot_frame = plot_frame.dropna(
    subset=["mean_normalized_payoff", "random_activation_weight"]
)

coverage = (
    plot_frame[plot_frame["population_group"] == "All agents"]
    .groupby(["n_agents", "best_response_count"])
    .size()
)
print(f"Samples per realized composition: {coverage.min()} to {coverage.max()}")

In [ ]:
def centers_to_edges(values, *, lower=None, upper=None):
    values = np.asarray(values, dtype=float)
    midpoints = (values[:-1] + values[1:]) / 2
    edges = np.concatenate((
        [values[0] - (midpoints[0] - values[0])],
        midpoints,
        [values[-1] + (values[-1] - midpoints[-1])],
    ))
    if lower is not None:
        edges[0] = lower
    if upper is not None:
        edges[-1] = upper
    return edges


def payoff_surface(panel, counts, weight_grid, *, kernel_sigma=0.085):
    surface = np.full((weight_grid.size, counts.size), np.nan)
    population_size = int(panel["n_agents"].iat[0])
    for column, count in enumerate(counts):
        series = panel[panel["best_response_count"] == count]
        if series.empty:
            continue
        x = series["random_activation_weight"].to_numpy(dtype=float)
        y = series["mean_normalized_payoff"].to_numpy(dtype=float)
        if count in (0, population_size):
            surface[:, column] = np.mean(y)
            continue
        distances = (weight_grid[:, None] - x[None, :]) / kernel_sigma
        weights = np.exp(-0.5 * distances**2)
        surface[:, column] = (weights @ y) / weights.sum(axis=1)
    return surface


def build_surfaces(frame, *, weight_points=160, kernel_sigma=0.085):
    weight_grid = np.linspace(
        frame["random_activation_weight"].min(),
        frame["random_activation_weight"].max(),
        weight_points,
    )
    surfaces = {}
    for n_agents_value in np.sort(frame["n_agents"].unique()):
        population = frame[frame["n_agents"] == n_agents_value]
        counts = np.sort(population["best_response_count"].unique()).astype(int)
        for group in population_groups:
            panel = population[population["population_group"] == group]
            surface = payoff_surface(
                panel, counts, weight_grid, kernel_sigma=kernel_sigma
            )
            if group == "Payoff-guided agents":
                surface[:, counts == 0] = np.nan
            elif group == "Random-action agents":
                surface[:, counts == n_agents_value] = np.nan
            surfaces[(n_agents_value, group)] = (counts, surface)
    return weight_grid, surfaces

In [ ]:
GAUSSIAN_SIGMA = 0.06
weight_grid, surfaces = build_surfaces(
    plot_frame, kernel_sigma=GAUSSIAN_SIGMA
)
kernel_center = float((weight_grid[0] + weight_grid[-1]) / 2)
agent_counts = np.sort(plot_frame["n_agents"].unique())
norm = Normalize(vmin=0.0, vmax=1.0)
base_cmap = plt.get_cmap("jet")
cmap = base_cmap.with_extremes(under=base_cmap(0.0))
fig, axes = plt.subplots(
    len(agent_counts),
    len(population_groups),
    figsize=(13, 11),
    sharex=True,
    sharey="row",
    gridspec_kw={"wspace": 0.06, "hspace": 0.06},
    squeeze=False,
)
fig.subplots_adjust(left=0.10, right=0.752, bottom=0.11, top=0.88)
weight_edges = centers_to_edges(weight_grid)
mesh = None
for row, n_agents_value in enumerate(agent_counts):
    for column, group in enumerate(population_groups):
        ax = axes[row, column]
        counts, surface = surfaces[(n_agents_value, group)]
        count_edges = centers_to_edges(
            counts, lower=-0.5, upper=float(n_agents_value) + 0.5
        )
        mesh = ax.pcolormesh(
            weight_edges,
            count_edges,
            surface.T,
            shading="flat",
            cmap=cmap,
            norm=norm,
            rasterized=True,
        )
        ax.set_xlim(weight_grid[0], weight_grid[-1])
        ax.set_ylim(count_edges[0], count_edges[-1])
        ax.set_box_aspect(1)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
        ax.grid(False)
        ax.tick_params(axis="both", which="both", length=0)

strip_facecolor = "#f2f2f2"
strip_edgecolor = "#b7b7b7"
for column, group in enumerate(population_groups):
    position = axes[0, column].get_position()
    strip = fig.add_axes([position.x0, position.y1 + 0.012, position.width, 0.04])
    strip.set_facecolor(strip_facecolor)
    strip.text(0.5, 0.5, group, ha="center", va="center", fontsize=18)
    strip.set_xticks([])
    strip.set_yticks([])
    for spine in strip.spines.values():
        spine.set_color(strip_edgecolor)
        spine.set_linewidth(0.8)

for row, n_agents_value in enumerate(agent_counts):
    position = axes[row, 0].get_position()
    strip = fig.add_axes([0.035, position.y0, 0.03, position.height])
    strip.set_facecolor(strip_facecolor)
    strip.text(
        0.5,
        0.5,
        f"{int(n_agents_value)} agents",
        ha="center",
        va="center",
        rotation=90,
        fontsize=16,
    )
    strip.set_xticks([])
    strip.set_yticks([])
    for spine in strip.spines.values():
        spine.set_color(strip_edgecolor)
        spine.set_linewidth(0.8)

fig.text(0.426, 0.07, "Random-agent activation weight", ha="center", fontsize=16)
fig.text(
    0.012,
    0.495,
    "Payoff-guided agent count",
    ha="center",
    va="center",
    rotation=90,
    fontsize=16,
)

assert mesh is not None
colorbar_axis = fig.add_axes([0.80, 0.31, 0.025, 0.49])
colorbar_axis.set_facecolor(cmap(0.0))
colorbar_axis.imshow(
    np.linspace(0.0, 1.0, 512)[:, None],
    cmap=cmap,
    vmin=0.0,
    vmax=1.0,
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    extent=(0.0, 1.0, 0.0, 1.0),
)
colorbar_axis.set_xlim(0.0, 1.0)
colorbar_axis.set_ylim(-0.16, 1.0)
colorbar_axis.set_xticks([])
colorbar_axis.set_yticks(
    [0.0, 1.0],
    labels=["Empty (0.00)", "Star (1.00)"],
)
colorbar_axis.minorticks_off()
colorbar_axis.yaxis.tick_right()
colorbar_axis.set_title(
    "Mean normalized\npayoff",
    fontsize=14,
    pad=6,
)
for spine in colorbar_axis.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

panel_width = axes[-1, -1].get_position().width
colorbar_position = colorbar_axis.get_position()
kernel_left = colorbar_position.x0 + colorbar_position.width / 2 - panel_width / 2
kernel_axis = fig.add_axes([kernel_left, 0.12, panel_width, 0.15])
kernel_x = np.linspace(
    kernel_center - 3 * GAUSSIAN_SIGMA,
    kernel_center + 3 * GAUSSIAN_SIGMA,
    200,
)
kernel_distance = (kernel_x - kernel_center) / GAUSSIAN_SIGMA
kernel_weight = np.exp(-0.5 * kernel_distance**2)
kernel_axis.fill_between(kernel_x, kernel_weight, color="black", alpha=0.12)
kernel_axis.plot(kernel_x, kernel_weight, color="black", linewidth=1.2)
kernel_axis.vlines(
    [kernel_center - GAUSSIAN_SIGMA, kernel_center + GAUSSIAN_SIGMA],
    0.0,
    np.exp(-0.5),
    color="black",
    linestyle=":",
    linewidth=0.8,
)
kernel_axis.text(
    kernel_center,
    -0.24,
    f"Gaussian kernel\nsigma = {GAUSSIAN_SIGMA:.3f}",
    ha="center",
    va="top",
    fontsize=11,
)
kernel_axis.set_xlim(weight_grid[0], weight_grid[-1])
kernel_axis.set_ylim(-0.48, 1.05)
kernel_axis.axis("off")
plt.show()

In [ ]:
run_id = result_store_path.stem
figure_output_dir = project_root / "artifacts/figures" / run_id
figure_output_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_output_dir / "best_response_proportion_effect.pdf"
fig.savefig(figure_path, format="pdf", bbox_inches="tight")
print(f"Saved: {figure_path.relative_to(project_root)}")